# LLM Ablation Study — Interactive Results

End-to-end walkthrough of a controlled char-level GPT experiment on TinyStories.

**Phases:**
1. Ablation matrix — 27 single-variable ablations against a fixed baseline
2. Seed sweep — multi-seed statistical validation (bootstrap CI)
3. Composition — stack confirmed improvements into `recipe_v1`
4. Scaling — apply the best recipe from 0.22 M → 21 M params

**Usage:** Run cells top-to-bottom. All data is loaded from `results/`.
Checkpoints are in `checkpoints/` (gitignored — needed only for the live generation cell).


In [ ]:
import json, math, os, random
from collections import defaultdict
from statistics import mean, stdev

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, HTML

plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

# ── helpers ──────────────────────────────────────────────────────────────────

def load_jsonl(path):
    if not os.path.exists(path):
        print(f"⚠  {path} not found — run the relevant script first")
        return []
    rows = []
    with open(path) as f:
        for line in f:
            s = line.strip()
            if s:
                try:
                    rows.append(json.loads(s))
                except json.JSONDecodeError:
                    pass
    return rows

def bootstrap_ci(vals, n_resample=2000, alpha=0.05, seed=42):
    if len(vals) < 2:
        return (mean(vals), float("nan"), float("nan")) if vals else (float("nan"),)*3
    rng = random.Random(seed)
    n = len(vals)
    resampled = sorted(mean([vals[rng.randrange(n)] for _ in range(n)]) for _ in range(n_resample))
    lo, hi = int(alpha / 2 * n_resample), int((1 - alpha / 2) * n_resample)
    return mean(vals), resampled[lo], resampled[min(hi, n_resample - 1)]

print("Setup complete ✓")


---
## Phase 1 — Ablation Matrix

27 ablations, each changing exactly one variable from the `small()` baseline
(1.22 M params, lr=3e-3, GELU, LayerNorm, learned position embeddings).
Single seed (1337). Differences below ~0.01 val loss are noise at n=1.


In [ ]:
ablation_rows = load_jsonl("results/ablation_results.jsonl")

ok       = [r for r in ablation_rows if not r.get("error") and not r.get("diverged")]
diverged = [r for r in ablation_rows if r.get("diverged") and not r.get("error")]
baseline_row = next((r for r in ok if r["name"] == "baseline"), None)
BASE_VAL = baseline_row["final"]["val_loss"]

print(f"Loaded: {len(ok)} ok | {len(diverged)} diverged | "
      f"{len(ablation_rows)-len(ok)-len(diverged)} errored")
print(f"Baseline  val_loss = {BASE_VAL:.4f}  bpc = {BASE_VAL/math.log(2):.4f}  "
      f"params = {baseline_row['n_params']/1e6:.2f}M")

records = [{
    "name":     r["name"],
    "category": r["category"],
    "val_loss": r["final"]["val_loss"],
    "bpc":      r["final"]["bpc"],
    "Δ val":    r["final"]["val_loss"] - BASE_VAL,
    "params M": round(r["n_params"] / 1e6, 3),
    "time s":   round(r["wallclock_seconds"]),
} for r in ok]

df_abl = (pd.DataFrame(records)
            .sort_values("val_loss")
            .reset_index(drop=True))
df_abl.index += 1

def _color_delta(v):
    if v < -0.01: return "color: #2ca02c; font-weight: bold"
    if v >  0.01: return "color: #d62728"
    return "color: #888888"

(df_abl.style
       .map(_color_delta, subset=["Δ val"])
       .format({"val_loss": "{:.4f}", "bpc": "{:.4f}",
                "Δ val": "{:+.4f}", "params M": "{:.3f}", "time s": "{}s"})
       .set_caption("All ablations ranked by val_loss ↓  (green = improvement, red = regression)"))


In [ ]:
non_base = df_abl[df_abl["name"] != "baseline"].copy().sort_values("Δ val")

colors = [
    "#2ca02c" if d < -0.01 else "#d62728" if d > 0.01 else "#aec7e8"
    for d in non_base["Δ val"]
]

fig, ax = plt.subplots(figsize=(10, 8))
bars = ax.barh(non_base["name"], non_base["Δ val"],
               color=colors, edgecolor="white", linewidth=0.4)
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("Δ val_loss vs baseline  (negative = better)")
ax.set_title("Ablation Δ val_loss from baseline\n"
             "green = improvement  ·  red = regression  ·  gray = within noise")
ax.grid(True, axis="x", alpha=0.3)

for bar, val in zip(bars, non_base["Δ val"]):
    pad = 0.003
    ax.text(
        val + (pad if val >= 0 else -pad),
        bar.get_y() + bar.get_height() / 2,
        f"{val:+.3f}", va="center",
        ha="left" if val >= 0 else "right", fontsize=8,
    )

plt.tight_layout()
plt.show()


In [ ]:
# Show baseline, top candidates, and the worst (non-diverged) run
highlight = {
    "baseline":     ("--", 2.5, "#555555"),
    "swiglu":       ("-",  2.2, "#2ca02c"),
    "lr_1e-2":      ("-",  2.0, "#1f77b4"),
    "qk_norm":      ("-",  2.0, "#ff7f0e"),
    "embd_std_0.1": ("-",  1.2, "#9467bd"),
    "relu2":        ("-",  1.2, "#8c564b"),
    "lr_1e-4":      (":",  1.2, "#d62728"),   # worst non-diverged
}
row_map = {r["name"]: r for r in ok}

fig, ax = plt.subplots(figsize=(10, 5))
for name, (ls, lw, color) in highlight.items():
    r = row_map.get(name)
    if r is None:
        continue
    curve = r["loss_curve"]
    delta = r["final"]["val_loss"] - BASE_VAL
    ax.plot(curve["steps"], curve["val_loss"],
            label=f"{name}  (Δ {delta:+.3f})",
            linestyle=ls, linewidth=lw, color=color)

ax.set_xlabel("Training step")
ax.set_ylabel("Val loss")
ax.set_title("Validation loss curves — selected ablations")
ax.legend(fontsize=9, bbox_to_anchor=(1.01, 1), loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## Phase 2 — Seed Sweep: Are the Effects Real?

Single-seed results are *candidates*, not conclusions. Here we re-run the top-3
candidates across 5 seeds (0–4) and compute **paired** bootstrap CIs on the
per-seed Δ val_loss. If the CI excludes zero the effect is likely real.

Baseline seed-to-seed sd ≈ 0.006 — differences below this are pure noise.


In [ ]:
sweep_rows = load_jsonl("results/seed_sweep_results.jsonl")

by_name = defaultdict(dict)
for r in sweep_rows:
    if not r.get("error") and not r.get("diverged"):
        by_name[r["name"]][r["seed"]] = r["final"]["val_loss"]

baseline_seeds = by_name.get("baseline", {})

stats = []
for name, seed_vals in by_name.items():
    vals = sorted(seed_vals.values())
    paired_deltas = [
        seed_vals[s] - baseline_seeds[s]
        for s in seed_vals if s in baseline_seeds
    ]
    m, lo, hi = bootstrap_ci(vals)
    dm, dlo, dhi = bootstrap_ci(paired_deltas) if len(paired_deltas) >= 2 else (float("nan"),)*3
    real = (len(paired_deltas) >= 3 and name != "baseline"
            and (dhi < 0 or dlo > 0))
    stats.append({
        "name": name, "n": len(vals),
        "mean val": m, "val CI lo": lo, "val CI hi": hi,
        "mean Δ": dm, "Δ CI lo": dlo, "Δ CI hi": dhi,
        "CI excl 0": "✓ yes" if real else "—",
        "likely_real": real,
        "paired_deltas": paired_deltas,
        "seed_vals": seed_vals,
    })

df_sweep = pd.DataFrame(stats).sort_values("mean val").reset_index(drop=True)

# Pretty table (drop internal columns)
display_cols = ["name", "n", "mean val", "mean Δ", "CI excl 0"]
(df_sweep[display_cols].style
    .format({"mean val": "{:.4f}", "mean Δ": "{:+.4f}"})
    .apply(lambda col: [
        "color: #2ca02c; font-weight: bold" if (r == "✓ yes" and df_sweep.loc[i, "mean Δ"] < 0)
        else "" for i, r in enumerate(col)
    ], subset=["CI excl 0"])
    .set_caption("Seed sweep results — paired bootstrap 95% CI"))


In [ ]:
non_base_stats = [s for s in stats if s["name"] != "baseline"]
non_base_stats.sort(key=lambda s: s["mean Δ"])

fig, ax = plt.subplots(figsize=(9, max(3.5, 0.65 * len(non_base_stats))))

for i, s in enumerate(non_base_stats):
    real, dm, dlo, dhi = s["likely_real"], s["mean Δ"], s["Δ CI lo"], s["Δ CI hi"]
    color = "#2ca02c" if (real and dm < 0) else "#d62728" if (real and dm > 0) else "#888888"
    ax.errorbar(dm, i,
                xerr=[[dm - dlo], [dhi - dm]],
                fmt="o", color=color, ecolor=color,
                markersize=9, capsize=5, linewidth=2)
    for d in s["paired_deltas"]:
        ax.plot(d, i, "o", color=color, alpha=0.3, markersize=5)

ax.axvline(0, color="black", linewidth=1)
ax.set_yticks(range(len(non_base_stats)))
ax.set_yticklabels([s["name"] for s in non_base_stats])
ax.invert_yaxis()
ax.set_xlabel("Paired Δ val_loss  (ablation − baseline, same seed)")
ax.set_title("Forest plot — 95% bootstrap CI of paired Δ\n"
             "green = confirmed improvement  ·  red = confirmed regression  ·  gray = no effect detected")
ax.grid(True, axis="x", alpha=0.3)

patches = [
    mpatches.Patch(color="#2ca02c", label="CI excludes 0 (improvement)"),
    mpatches.Patch(color="#d62728", label="CI excludes 0 (regression)"),
    mpatches.Patch(color="#888888", label="CI includes 0 (no effect detected)"),
]
ax.legend(handles=patches, fontsize=9, loc="lower right")
plt.tight_layout()
plt.show()


---
## Phase 3a — Composition: Stacking the Winners

`recipe_v1` = `swiglu` + `qk_norm` + `lr=1e-2`

Do the three improvements add up? Perfect additivity would give Δ ≈ −0.098.
The actual composition Δ tells us how much is genuinely compounding vs. shared.


In [ ]:
components = ["lr_1e-2", "swiglu", "qk_norm"]
all_shown  = ["baseline"] + components + ["recipe_v1"]

comp = {s["name"]: s for s in stats if s["name"] in all_shown}
valid = [n for n in all_shown if n in comp]

# ── left: mean val loss bar chart ────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

means = [comp[n]["mean val"] for n in valid]
lo    = [comp[n]["val CI lo"] for n in valid]
hi    = [comp[n]["val CI hi"] for n in valid]
pal   = ["#aec7e8" if n == "baseline" else
         "#2ca02c"  if n == "recipe_v1" else
         "#1f77b4"  for n in valid]

ax1.bar(valid, means, color=pal,
        yerr=[[m - l for m, l in zip(means, lo)],
              [h - m for m, h in zip(means, hi)]],
        capsize=5, error_kw={"linewidth": 1.5}, width=0.55)
ax1.set_ylabel("Mean val_loss (↓ better)")
ax1.set_title("Mean val_loss across 5 seeds\n(error bars = 95% CI)")
ax1.set_ylim(min(lo) - 0.012, max(hi) + 0.012)
ax1.tick_params(axis="x", rotation=20)
ax1.axhline(comp["baseline"]["mean val"], color="#aec7e8",
            linestyle="--", linewidth=1, alpha=0.7)

# ── right: additivity check ───────────────────────────────────────────────────
ind_deltas = {n: comp[n]["mean Δ"] for n in components if n in comp}
expected   = sum(ind_deltas.values())
actual     = comp["recipe_v1"]["mean Δ"] if "recipe_v1" in comp else float("nan")

labels = ["Sum of\nindividual Δ", "recipe_v1\nactual Δ"]
vals   = [expected, actual]
colors_bar = ["#ff7f0e", "#2ca02c"]
bars = ax2.bar(labels, vals, color=colors_bar, width=0.45)
ax2.axhline(0, color="black", linewidth=0.8)
ax2.set_ylabel("Δ val_loss vs baseline")
pct = actual / expected * 100 if expected else float("nan")
ax2.set_title(f"Additivity check\nexpected {expected:.3f}  ·  actual {actual:.3f}  ·  {pct:.0f}% additive")
for bar, v in zip(bars, vals):
    ax2.text(bar.get_x() + bar.get_width()/2, v - 0.002, f"{v:.3f}",
             ha="center", va="top", fontsize=10, fontweight="bold", color="white")
ax2.set_ylim(min(vals) - 0.01, 0.005)

plt.suptitle("recipe_v1 = swiglu + qk_norm + lr=1e-2", fontsize=11, y=1.02)
plt.tight_layout()
plt.show()

print(f"Individual deltas: {ind_deltas}")
print(f"Expected (fully additive): {expected:.4f}")
print(f"Actual recipe_v1 delta:    {actual:.4f}  ({pct:.0f}% additive)")


---
## Phase 3b — Scaling Study

`recipe_v1` applied at 4 model sizes. Fixed compute budget per point:
**4000 steps × 64 batch × 256 block = 65.5 M tokens/run**, seed = 1337.

This is a fixed-compute scaling curve — all models see the same number of tokens,
only the parameter count changes.


In [ ]:
TOKENS = 4000 * 64 * 256  # fixed per run

scaling_rows = load_jsonl("results/scale_study_results.jsonl")
scaling_ok   = sorted([r for r in scaling_rows if not r.get("error")],
                      key=lambda r: r["n_params"])

if not scaling_ok:
    print("⚠  No scaling results found. Run  python scale_study.py  first.")
else:
    df_scale = pd.DataFrame([{
        "config":   r["name"],
        "params M": round(r["n_params"] / 1e6, 3),
        "val loss": r["final"]["val_loss"],
        "bpc":      r["final"]["bpc"],
        "FLOPs":    6 * r["n_params"] * TOKENS,
        "time s":   round(r["wallclock_seconds"]),
    } for r in scaling_ok])

    display(
        df_scale.style
        .format({"params M": "{:.3f}M", "val loss": "{:.4f}",
                 "bpc": "{:.4f}", "FLOPs": "{:.2e}", "time s": "{}s"})
        .background_gradient(subset=["val loss"], cmap="RdYlGn_r")
        .set_caption("Scaling study — recipe_v1 at each model size (seed=1337, single run)")
    )


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

params = np.array([r["n_params"] for r in scaling_ok])
vals   = np.array([r["final"]["val_loss"] for r in scaling_ok])
bpcs   = np.array([r["final"]["bpc"] for r in scaling_ok])
flops  = np.array([6 * r["n_params"] * TOKENS for r in scaling_ok])
labels = [r["name"] for r in scaling_ok]
sizes  = [r["n_params"]/1e6 for r in scaling_ok]

for ax, x, xlabel in [
    (axes[0], params, "Parameters"),
    (axes[1], flops,  "Estimated FLOPs  (6·N·D)"),
]:
    ax.plot(x, vals, "o-", color="#2ca02c", linewidth=2.5, markersize=11, zorder=3)
    ax.axhline(1.0, color="#888", linestyle="--", linewidth=1, label="BPC = 1.0 (val loss = ln 2)")
    for xi, vi, ni, si in zip(x, vals, labels, sizes):
        ax.annotate(f"{ni}\n({si:.2f}M params)",
                    (xi, vi), textcoords="offset points",
                    xytext=(10, 4), fontsize=8.5, color="#333")
    ax.set_xscale("log")
    ax.set_xlabel(f"{xlabel}  (log scale)")
    ax.set_ylabel("Val loss  (↓ better)")
    ax.set_title(f"recipe_v1: val loss vs {xlabel}")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle("Fixed compute budget per point: 65.5 M tokens/run  ·  seed = 1337",
             y=1.02, fontsize=10, color="#555")
plt.tight_layout()
plt.show()

# BPC annotation
print("BPC breakdown:")
for r in scaling_ok:
    bpc = r["final"]["bpc"]
    flag = " ← below 1.0 bit/char" if bpc < 1.0 else ""
    print(f"  {r['name']:20s}  {r['n_params']/1e6:.2f}M params  bpc={bpc:.4f}{flag}")


---
## Generation Samples — What Does the Text Look Like?

Samples are stored inside each JSONL result row (no checkpoint needed to view these).
We compare: baseline vs top individual ablations, and then the full scaling ladder.


In [ ]:
row_map_all = {r["name"]: r for r in ok}
# also include scaling configs
for r in scaling_ok:
    row_map_all[r["name"]] = r

def get_sample(name, prompt_idx=0):
    r = row_map_all.get(name)
    if r is None:
        return None, None, None
    gen = r.get("generation", {})
    samples = gen.get("samples", [])
    if prompt_idx >= len(samples):
        return None, None, None
    s = samples[prompt_idx]
    # completion already includes the prompt; strip it
    text = s["completion"][len(s["prompt"]):]
    return s["prompt"], text[:450], r["final"]["val_loss"]

def side_by_side(names, prompt_idx=0, title=""):
    prompt = None
    cols = []
    for name in names:
        p, text, val = get_sample(name, prompt_idx)
        if text is None:
            continue
        if prompt is None:
            prompt = p
        cols.append((name, val, text))

    if not cols:
        print("No samples found.")
        return

    width = max(40, 95 // len(cols))
    header_line = "  ".join(f"{name!s:^{width}}" for name, _, _ in cols)
    val_line    = "  ".join(f"val={val:.4f}".center(width) for _, val, _ in cols)
    sep         = ("─" * width + "  ") * len(cols)

    if title:
        print(f"\n{'━'*80}\n{title}\n{'━'*80}")
    print(f'Prompt: "{prompt}"')
    print(sep)
    print(header_line)
    print(val_line)
    print(sep)

    # interleave lines
    texts = [t.splitlines() for _, _, t in cols]
    max_lines = max(len(t) for t in texts)
    for i in range(max_lines):
        row = "  ".join(
            (t[i] if i < len(t) else "").ljust(width)[:width]
            for t in texts
        )
        print(row)
    print()

# ── Ablation comparison ───────────────────────────────────────────────────────
side_by_side(
    ["baseline", "swiglu", "qk_norm", "lr_1e-2"],
    prompt_idx=0,
    title="Ablation comparison — baseline vs top improvements  (Prompt 1)",
)
side_by_side(
    ["baseline", "swiglu", "qk_norm", "lr_1e-2"],
    prompt_idx=2,
    title="Ablation comparison — baseline vs top improvements  (Prompt 3)",
)


In [ ]:
# ── Scaling comparison ────────────────────────────────────────────────────────
scale_names = [r["name"] for r in scaling_ok]

side_by_side(
    scale_names,
    prompt_idx=0,
    title="Scaling ladder — micro → small → medium → large  (Prompt 1)",
)
side_by_side(
    scale_names,
    prompt_idx=1,
    title="Scaling ladder — micro → small → medium → large  (Prompt 2)",
)


---
## Live Generation (optional)

Requires checkpoint files in `checkpoints/`. These are gitignored — run the training
scripts first, or skip this cell if you just want to view the pre-recorded samples above.


In [ ]:
import sys
sys.path.insert(0, ".")
import torch
import ablate

CHECKPOINT = "checkpoints/large_recipe_seed1337.pt"  # change as needed
PROMPTS = ["Once upon a time", "The little girl", "One day"]

if not os.path.exists(CHECKPOINT):
    print(f"⚠  {CHECKPOINT} not found. Run scale_study.py to generate it,")
    print("   or change CHECKPOINT to an existing file (python infer.py --list).")
else:
    ckpt = torch.load(CHECKPOINT, map_location="cpu", weights_only=False)
    cfg  = ablate.Config(**ckpt["config"])
    model = ablate.GPT(cfg)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    stoi, itos = ckpt["stoi"], ckpt["itos"]

    print(f"Loaded: {CHECKPOINT}")
    print(f"  params  = {sum(p.numel() for p in model.parameters()):,}")
    print(f"  val_loss (saved) = {ckpt['final']['val_loss']:.4f}")
    print()

    temperature = 0.8   # adjust: lower = more focused, higher = more random
    top_k       = 40

    for prompt in PROMPTS:
        ids = torch.tensor([[stoi.get(c, 0) for c in prompt]], dtype=torch.long)
        with torch.no_grad():
            out = model.generate(ids, max_new_tokens=300,
                                 temperature=temperature, top_k=top_k)
        text = "".join(itos[i] for i in out[0].tolist())
        print(f'── Prompt: "{prompt}" ──')
        print(text)
        print()


---
## Key Takeaways

| Finding | Detail |
|---|---|
| **SwiGLU beats GELU** | Δ −0.034 (CI: [−0.038, −0.029]) — confirmed across 5 seeds |
| **QK-norm helps** | Δ −0.029 (CI: [−0.035, −0.020]) — stabilises attention at scale |
| **lr=1e-2 works** | Δ −0.036 (CI: [−0.043, −0.029]) — cosine decay rescues the high LR |
| **Partial additivity** | Stack gives −0.052 vs predicted −0.098 (~53%) — some shared gain |
| **Clean scaling** | val loss drops monotonically: 0.86 → 0.71 → 0.68 → 0.63 over 0.2–21M params |
| **BPC < 1.0 at 6M** | The recipe crosses the 1 bit/char threshold at medium scale |
| **No overfitting** | Train/val gap ≈ 0.007 throughout — model is capacity-bound, not data-bound |
| **post_norm diverges** | val = 3.06 — pre-norm is load-bearing without extra stabilisation |

**Next steps:** upload best checkpoint to Hugging Face · run at 50 M+ params with more data · re-tune lr per scale
